## House Prices - Advanced Regression Techniques
- With 79 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this competition challenges you to predict the final price of each home.
- Dataset Source - https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data
  
This notebook is for recording what I’ve learned and implemented in my machine learning journey.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv


In [2]:
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_df  = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
train_df.shape

(1460, 81)

## Data Check & Processing
- check missing values
- One Hot Encoding for Categorical features
- change data type

In [4]:
with pd.option_context('display.max_rows', None):
    print(train_df.isna().sum().sort_values(ascending=False))

PoolQC           1453
MiscFeature      1406
Alley            1369
Fence            1179
MasVnrType        872
FireplaceQu       690
LotFrontage       259
GarageQual         81
GarageFinish       81
GarageType         81
GarageYrBlt        81
GarageCond         81
BsmtFinType2       38
BsmtExposure       38
BsmtCond           37
BsmtQual           37
BsmtFinType1       37
MasVnrArea          8
Electrical          1
Condition2          0
BldgType            0
Neighborhood        0
LandSlope           0
LotConfig           0
Condition1          0
LandContour         0
LotShape            0
Street              0
LotArea             0
MSSubClass          0
MSZoning            0
Id                  0
Utilities           0
HouseStyle          0
Foundation          0
ExterQual           0
ExterCond           0
BsmtUnfSF           0
TotalBsmtSF         0
Heating             0
BsmtFinSF1          0
Exterior2nd         0
Exterior1st         0
RoofMatl            0
RoofStyle           0
YearRemodA

In [5]:
train_df = train_df.dropna(subset=['Electrical']).reset_index(drop=True)  ##Drop one record whose Electrical value is missing

In [6]:
train_df.loc[train_df['MasVnrArea'].isna(), ['MasVnrType', 'MasVnrArea']]
## records whose MasVnrArea is NA also have NA for MasVnrType

,MasVnrType,MasVnrArea
234,NaN,NaN
529,NaN,NaN
650,NaN,NaN
936,NaN,NaN
973,NaN,NaN
977,NaN,NaN
1243,NaN,NaN
1278,NaN,NaN


In [7]:
train_df['MasVnrArea'] = train_df['MasVnrArea'].fillna(0)   ## Fill with 0

In [8]:
## Bsmt related features
train_df.loc[
    train_df['BsmtFinType2'].isna(),
    train_df.columns[train_df.columns.str.contains('Bsmt')]
]

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,BsmtFullBath,BsmtHalfBath
17,NaN,NaN,NaN,NaN,0,NaN,0,0,0,0,0
39,NaN,NaN,NaN,NaN,0,NaN,0,0,0,0,0
90,NaN,NaN,NaN,NaN,0,NaN,0,0,0,0,0
102,NaN,NaN,NaN,NaN,0,NaN,0,0,0,0,0
156,NaN,NaN,NaN,NaN,0,NaN,0,0,0,0,0
182,NaN,NaN,NaN,NaN,0,NaN,0,0,0,0,0
259,NaN,NaN,NaN,NaN,0,NaN,0,0,0,0,0
332,Gd,TA,No,GLQ,1124,NaN,479,1603,3206,1,0
342,NaN,NaN,NaN,NaN,0,NaN,0,0,0,0,0
362,NaN,NaN,NaN,NaN,0,NaN,0,0,0,0,0


In [9]:
##this record has other information about basement but only this feature value is missing
train_df.loc[332, 'BsmtFinType2'] = 'Unf'

no_bsmt = train_df['TotalBsmtSF'] == 0
train_df.loc[no_bsmt, ['BsmtQual', 'BsmtCond', 'BsmtFinType1', 'BsmtFinType2', 'BsmtExposure']] = 'None'
train_df['BsmtExposure'] = train_df['BsmtExposure'].fillna('No')  ## Fill other NA records

In [10]:
## Garage
with pd.option_context('display.max_rows', None):
    print(train_df.loc[
    train_df['GarageQual'].isna(),
    train_df.columns[train_df.columns.str.contains('Garage')]
    ])

     GarageType  GarageYrBlt GarageFinish  GarageCars  GarageArea GarageQual  \
39          NaN          NaN          NaN           0           0        NaN   
48          NaN          NaN          NaN           0           0        NaN   
78          NaN          NaN          NaN           0           0        NaN   
88          NaN          NaN          NaN           0           0        NaN   
89          NaN          NaN          NaN           0           0        NaN   
99          NaN          NaN          NaN           0           0        NaN   
108         NaN          NaN          NaN           0           0        NaN   
125         NaN          NaN          NaN           0           0        NaN   
127         NaN          NaN          NaN           0           0        NaN   
140         NaN          NaN          NaN           0           0        NaN   
148         NaN          NaN          NaN           0           0        NaN   
155         NaN          NaN          Na

In [11]:
no_garage = train_df['GarageArea'] == 0
train_df.loc[no_garage, ['GarageQual', 'GarageFinish', 'GarageType', 'GarageCond']] = 'None'
train_df.loc[no_garage, 'GarageYrBlt'] = train_df.loc[no_garage, 'YearBuilt']

In [12]:
# Fill missing LotFrontage values with the median value of the corresponding Neighborhood
train_df['LotFrontage'] = (
    train_df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
)

In [13]:
# Fireplace
train_df.loc[
    train_df['FireplaceQu'].isna(),
    train_df.columns[train_df.columns.str.contains('Fireplace')]
]

,Fireplaces,FireplaceQu
0,0,NaN
5,0,NaN
10,0,NaN
12,0,NaN
15,0,NaN
...,...,...
1451,0,NaN
1452,0,NaN
1453,0,NaN
1457,0,NaN


In [14]:
train_df['FireplaceQu'] = train_df['FireplaceQu'].fillna('None')

In [15]:
# MasVnrType
print(train_df.loc[
    train_df['MasVnrType'].isna(),
    train_df.columns[train_df.columns.str.contains('MasVnr')]
])

     MasVnrType  MasVnrArea
1           NaN         0.0
3           NaN         0.0
5           NaN         0.0
8           NaN         0.0
9           NaN         0.0
...         ...         ...
1453        NaN         0.0
1454        NaN         0.0
1456        NaN         0.0
1457        NaN         0.0
1458        NaN         0.0

[871 rows x 2 columns]


In [16]:
train_df['MasVnrType'].value_counts(dropna=False)

MasVnrType
NaN        871
BrkFace    445
Stone      128
BrkCmn      15
Name: count, dtype: int64

In [17]:
#Fill with most frequent value if 'MasVnrArea' > 0
train_df.loc[(train_df['MasVnrArea'] > 0) & (train_df['MasVnrType'].isna()),'MasVnrType'] = 'BrkFace' 
train_df['MasVnrType'] = train_df['MasVnrType'].fillna('None')

In [18]:
#Drop unnecessary features
train_df = train_df.drop(['MiscFeature', 'Alley', 'Fence','PoolQC'], axis=1)
train_df = train_df.drop(['Id', 'Utilities', 'Street'], axis=1)

In [19]:
print(train_df.isna().sum().sort_values(ascending=False))

MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
LotShape         0
                ..
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
SalePrice        0
Length: 74, dtype: int64


In [20]:
print(train_df.dtypes)

MSSubClass         int64
MSZoning          object
LotFrontage      float64
LotArea            int64
LotShape          object
                  ...   
MoSold             int64
YrSold             int64
SaleType          object
SaleCondition     object
SalePrice          int64
Length: 74, dtype: object


In [21]:
#change 'MSSubClass' datatype to str because this feature is categorical feature
train_df['MSSubClass'] = train_df['MSSubClass'].astype(str)

In [22]:
#same process for test dataset
print(test_df.isna().sum().sort_values(ascending=False))

PoolQC           1456
MiscFeature      1408
Alley            1352
Fence            1169
MasVnrType        894
                 ... 
EnclosedPorch       0
MiscVal             0
MoSold              0
YrSold              0
SaleCondition       0
Length: 80, dtype: int64


In [23]:
test_df['MasVnrArea'] = test_df['MasVnrArea'].fillna(0)
test_df.loc[(test_df['MasVnrArea'] > 0) & (test_df['MasVnrType'].isna()),'MasVnrType'] = 'BrkFace'
test_df['MasVnrType'] = test_df['MasVnrType'].fillna('None')
test_df['FireplaceQu'] = test_df['FireplaceQu'].fillna('None')
test_df = test_df.drop(['MiscFeature', 'Alley', 'Fence','PoolQC'], axis=1)
test_df = test_df.drop(['Utilities', 'Street'], axis=1)
ids = test_df.pop('Id')  #Copy and drop test data Ids

In [24]:
no_bsmt = test_df['TotalBsmtSF'] == 0
test_df.loc[no_bsmt, ['BsmtQual', 'BsmtCond', 'BsmtFinType1', 'BsmtFinType2', 'BsmtExposure']] = 'None'

no_garage = test_df['GarageArea'] == 0
test_df.loc[no_garage, ['GarageQual', 'GarageFinish', 'GarageType', 'GarageCond']] = 'None'
test_df.loc[no_garage, 'GarageYrBlt'] = test_df.loc[no_garage, 'YearBuilt']
test_df['LotFrontage'] = (
    test_df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
)

In [25]:
test_df.loc[
    test_df['BsmtCond'].isna(),
    test_df.columns[test_df.columns.str.contains('Bsmt')]
]

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,BsmtFullBath,BsmtHalfBath
580,Gd,NaN,Mn,GLQ,1044.0,Rec,382.0,0.0,1426.0,1.0,0.0
660,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
725,TA,NaN,No,BLQ,1033.0,Unf,0.0,94.0,1127.0,0.0,1.0
1064,TA,NaN,Av,ALQ,755.0,Unf,0.0,240.0,995.0,0.0,0.0


In [26]:
test_df.loc[660, ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath','BsmtHalfBath']] = 0
test_df.loc[660, ['BsmtQual', 'BsmtCond', 'BsmtFinType1', 'BsmtFinType2', 'BsmtExposure']] = 'None'

In [27]:
train_X = train_df.drop('SalePrice', axis=1)
cat_cols = train_X.select_dtypes(include='object').columns
num_cols = train_X.select_dtypes(exclude='object').columns

num_fill_values = train_df[num_cols].median()   #Fill other missing numerical values form test data with median value
cat_fill_values = train_df[cat_cols].mode().iloc[0]  # Fill with Most frequent value
test_df[num_cols]  = test_df[num_cols].fillna(num_fill_values)
test_df[cat_cols]  = test_df[cat_cols].fillna(cat_fill_values)
test_df['MSSubClass'] = test_df['MSSubClass'].astype(str)

In [28]:
print(test_df.isna().sum().sort_values(ascending=False))

MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
LotShape         0
                ..
MiscVal          0
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
Length: 73, dtype: int64


In [29]:
#Encoding
y = train_df['SalePrice']
train_X = train_df.drop('SalePrice', axis=1)
n_train = train_X.shape[0]
full = pd.concat([train_X, test_df], axis=0)
cat_cols = full.select_dtypes(include='object').columns
full = pd.get_dummies(full, columns=cat_cols, drop_first=True)
train_df = full.iloc[:n_train]
test_df = full.iloc[n_train:]
print(train_df.shape)
print(test_df.shape)

(1459, 258)
(1459, 258)


In [30]:
train_df.head()

,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,...,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,65.0,8450,7,5,2003,2003,196.0,706.0,0.0,150.0,...,False,False,False,False,True,False,False,False,True,False
1,80.0,9600,6,8,1976,1976,0.0,978.0,0.0,284.0,...,False,False,False,False,True,False,False,False,True,False
2,68.0,11250,7,5,2001,2002,162.0,486.0,0.0,434.0,...,False,False,False,False,True,False,False,False,True,False
3,60.0,9550,7,5,1915,1970,0.0,216.0,0.0,540.0,...,False,False,False,False,True,False,False,False,False,False
4,84.0,14260,8,5,2000,2000,350.0,655.0,0.0,490.0,...,False,False,False,False,True,False,False,False,True,False


## Model Training

In [31]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

# separate dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(train_df,y,test_size=0.2,random_state=2)
X_train.shape, X_test.shape

((1167, 258), (292, 258))

In [32]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [33]:
## Model Training
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "Adaboost Regressor":AdaBoostRegressor(),
    "Graident BoostRegressor":GradientBoostingRegressor(),
    "Xgboost Regressor":XGBRegressor()
   
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    
    print('performance for Training set')
    print("- RMSE: {:.4f}".format(model_train_rmse))
    print("- MAE: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('---------------------------------')
    
    print('performance for Test set')
    print("- RMSE: {:.4f}".format(model_test_rmse))
    print("- MAE: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*30)
    print('\n')

Linear Regression
performance for Training set
- RMSE: 20686.0454
- MAE: 13455.3386
- R2 Score: 0.9336
---------------------------------
performance for Test set
- RMSE: 27736.9165
- MAE: 18384.4793
- R2 Score: 0.8665


Lasso
performance for Training set
- RMSE: 20696.0164
- MAE: 13471.9604
- R2 Score: 0.9336
---------------------------------
performance for Test set
- RMSE: 27260.2918
- MAE: 18174.5920
- R2 Score: 0.8711


Ridge
performance for Training set
- RMSE: 24561.8356
- MAE: 15663.0235
- R2 Score: 0.9064
---------------------------------
performance for Test set
- RMSE: 25949.6663
- MAE: 17675.8807
- R2 Score: 0.8832


K-Neighbors Regressor
performance for Training set
- RMSE: 38880.1511
- MAE: 24170.5858
- R2 Score: 0.7656
---------------------------------
performance for Test set
- RMSE: 44983.8753
- MAE: 29954.8103
- R2 Score: 0.6489


Decision Tree
performance for Training set
- RMSE: 0.0000
- MAE: 0.0000
- R2 Score: 1.0000
---------------------------------
performance for

In [34]:
#Parameter for Hyperparamter tuning

rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}

gradient_params={"loss": ['squared_error','huber','absolute_error'],
             "criterion": ['friedman_mse','squared_error','mse'],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500],
              "max_depth": [5, 8, 15, None, 10],
            }

# Models list for Hyperparameter tuning
randomcv_models = [
                   ("RF", RandomForestRegressor(), rf_params),
                   ("GradientBoost",GradientBoostingRegressor(),gradient_params)
                   
                   ]

In [35]:
##Hyperparameter Tuning
"""
This one takes long time.
"""
"""
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=4)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

"""

'\nfrom sklearn.model_selection import RandomizedSearchCV\n\nmodel_param = {}\nfor name, model, params in randomcv_models:\n    random = RandomizedSearchCV(estimator=model,\n                                   param_distributions=params,\n                                   n_iter=100,\n                                   cv=3,\n                                   verbose=2,\n                                   n_jobs=4)\n    random.fit(X_train, y_train)\n    model_param[name] = random.best_params_\n\nfor model_name in model_param:\n    print(f"---------------- Best Params for {model_name} -------------------")\n    print(model_param[model_name])\n\n'

In [36]:
# Best Parameter from above Cell. I used Gradient Boost
final_model = GradientBoostingRegressor(n_estimators= 500, min_samples_split=15, max_depth=8, loss= 'absolute_error', criterion='friedman_mse')
final_model.fit(train_df, y)
predictions_final = final_model.predict(test_df)
output = pd.DataFrame({'Id': ids, 'SalePrice': predictions_final})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
